# ECON 3P14 — Macro Theory with Python

This is **Notebook 2 of 3** in the ECON 3P14 Python companion series. We take the models you've seen in class — the **one-period** model from Module 1 and the **two-period** intertemporal model from Module 2 — and re-draw them in Python. Every graph you've sketched by hand (budget line, endowment point, tangency, policy experiments) becomes two or three lines of code you can tweak and re-run.

The goal is to build intuition: pick numbers, shift a lever, watch the equilibrium move. Nothing here is required calculation — this is a visualization sandbox.

**Pre-reqs.** Notebook 1 (Python basics + pandas). The only new tool here is `matplotlib` for drawing, which Notebook 1 introduced at the end.


## Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Default plot styling for the notebook.
plt.rcParams['figure.figsize'] = (6, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3


## Part A — The one-period model (brief recap)

In the **one-period** model the representative consumer chooses leisure $l$ and consumption $c$, subject to a time constraint $l + N^S = h$ (where $N^S$ is labor supplied and $h$ is total time) and a budget constraint $c = w \cdot N^S + \pi - T$ (where $w$ is the real wage, $\pi$ is firm profits distributed back to consumers, and $T$ are lump-sum taxes).

Rearranging gives the **budget line in $(l, c)$ space**:

$$c = w\,(h - l) + \pi - T$$

Slope is $-w$: giving up one hour of leisure buys $w$ extra units of consumption. The plot below draws the budget line and a Cobb-Douglas indifference curve for a fixed set of parameters.


In [ ]:
# One-period parameters.
h   = 24       # total available hours
w   = 10       # real wage
pi  = 20       # firm profit distributed back
T   = 5        # lump-sum tax

# Budget line in (l, c) space.
l_grid = np.linspace(0, h, 200)
c_bl   = w * (h - l_grid) + pi - T

# Cobb-Douglas U(c, l) = c^a * l^(1-a). With a = 0.5 the optimum is
# c* = 0.5 * (w*h + pi - T)   and   l* = 0.5 * (w*h + pi - T) / w
a = 0.5
M = w * h + pi - T              # full-income
c_star = a * M
l_star = (1 - a) * M / w
U_star = (c_star ** a) * (l_star ** (1 - a))

# Indifference curve through the optimum: c = (U/l^(1-a))^(1/a)
l_ic = np.linspace(2, h - 2, 200)
c_ic = (U_star / (l_ic ** (1 - a))) ** (1 / a)

fig, ax = plt.subplots()
ax.plot(l_grid, c_bl, lw=2, label=f'Budget line (slope $-w = -{w}$)')
ax.plot(l_ic, c_ic, lw=1.5, ls='--', label='Indifference curve')
ax.scatter(l_star, c_star, s=70, color='forestgreen', zorder=5,
           label=f'Optimum $(l^*, c^*) = ({l_star:.1f}, {c_star:.1f})$')
ax.set_xlim(0, h); ax.set_ylim(0, c_bl.max() * 1.1)
ax.set_xlabel('Leisure  $l$'); ax.set_ylabel('Consumption  $c$')
ax.set_title('One-period model: consumer equilibrium')
ax.legend(fontsize=9)
plt.show()


**Try it.** Change the wage `w` or the tax `T` in the cell above and rerun. A higher `w` makes the budget line steeper (each hour of work buys more consumption) and pushes the optimum down-and-right (less leisure, more consumption). A higher `T` shifts the whole budget line down in parallel.


## Part B — The two-period intertemporal model

In **Module 2** we add time. The consumer lives two periods, receives exogenous income $y$ and $y'$, pays taxes $t$ and $t'$, and chooses present consumption $c$ and future consumption $c'$ to maximize a utility function $U(c, c')$.

**Budget constraints.** With savings $s$ at interest rate $r$:

$$c + s = y - t \qquad c' = y' - t' + (1+r)s$$

Collapsing the two by substitution gives the **lifetime budget constraint**:

$$c + \frac{c'}{1 + r} \;=\; \underbrace{\,(y - t) + \frac{y' - t'}{1 + r}\,}_{\text{lifetime wealth } we}$$

In $(c, c')$ space this is a straight line of slope $-(1+r)$ passing through the **endowment point** $E = (y - t, y' - t')$, which is the zero-savings bundle ($s = 0$). Points to the left of $E$ are **savers / lenders** ($s > 0$); points to the right are **borrowers** ($s < 0$). The optimum $(c^*, c'^*)$ is the tangency of an indifference curve with this line.

The code below packages all of this into reusable helper functions so we can re-run the picture quickly under different policies.


In [ ]:
def lifetime_wealth(y, yp, t, tp, r):
    return (y - t) + (yp - tp) / (1 + r)

def endowment(y, yp, t, tp):
    return (y - t, yp - tp)

def budget_line(y, yp, t, tp, r, n=200):
    """Return (c, c') arrays describing the lifetime budget line."""
    we = lifetime_wealth(y, yp, t, tp, r)
    c = np.linspace(0, we, n)
    cp = we * (1 + r) - (1 + r) * c
    return c, cp

def indifference_curve(U_level, beta, c_min=5, c_max=200, n=200):
    """Cobb-Douglas U = ln c + beta * ln c'  ->  c' = exp((U - ln c)/beta)."""
    c = np.linspace(c_min, c_max, n)
    cp = np.exp((U_level - np.log(c)) / beta)
    return c, cp

def solve_optimum(y, yp, t, tp, r, beta):
    """Closed form for U = ln c + beta * ln c' subject to the lifetime BL."""
    we = lifetime_wealth(y, yp, t, tp, r)
    c_star  = we / (1 + beta)
    cp_star = (1 + r) * beta * we / (1 + beta)
    return c_star, cp_star

# Smoke-test.
y, yp, t, tp, r, beta = 100, 120, 10, 15, 0.10, 1 / 1.10
print('we =', lifetime_wealth(y, yp, t, tp, r))
print('E  =', endowment(y, yp, t, tp))
print('(c*, c\'*) =', solve_optimum(y, yp, t, tp, r, beta))


### Baseline equilibrium picture

Call the helpers to plot the budget line, endowment $E$, the indifference curve through the optimum, and the tangency.


In [ ]:
def plot_equilibrium(y, yp, t, tp, r, beta, ax=None, label_suffix=''):
    if ax is None:
        fig, ax = plt.subplots()
    c_bl, cp_bl = budget_line(y, yp, t, tp, r)
    E = endowment(y, yp, t, tp)
    c_star, cp_star = solve_optimum(y, yp, t, tp, r, beta)
    U_star = np.log(c_star) + beta * np.log(cp_star)
    c_ic, cp_ic = indifference_curve(U_star, beta, c_min=5, c_max=c_bl.max())
    ax.plot(c_bl, cp_bl, lw=2, label=f'Budget line {label_suffix}')
    ax.plot(c_ic, cp_ic, lw=1.2, ls='--', label=f'Indiff. curve {label_suffix}')
    ax.scatter(*E, s=60, color='crimson', zorder=5, label=f'$E{label_suffix}$')
    ax.scatter(c_star, cp_star, s=60, color='forestgreen', zorder=5,
               label=f'$(c^*, c\'^*){label_suffix}$')
    ax.set_xlabel('Present consumption $c$')
    ax.set_ylabel('Future consumption $c\'$')
    return ax

ax = plot_equilibrium(y=100, yp=120, t=10, tp=15, r=0.10, beta=1 / 1.10)
ax.set_title('Two-period consumer equilibrium — baseline')
ax.legend(fontsize=9)
plt.show()


## Part C — Experiment 1: an interest-rate hike (pivot through $E$)

When the interest rate rises from $r_1$ to $r_2$, the slope of the budget line steepens from $-(1 + r_1)$ to $-(1 + r_2)$. The line pivots **clockwise around the endowment point** $E$: at the endowment the consumer neither saves nor borrows, so bundle $E$ remains affordable at any $r$.

- **Borrower** (starts to the right of $E$): the feasible region shrinks, utility falls, $c$ unambiguously drops, savings rise (borrows less), future consumption is ambiguous.
- **Saver** (starts to the left of $E$): the feasible region expands, utility rises, $c'$ unambiguously rises, current consumption and savings ambiguous.

The plot below overlays the before and after lines so you can see the pivot.


In [ ]:
fig, ax = plt.subplots()
y, yp, t, tp = 100, 120, 10, 15

for r_val, ls in [(0.05, '-'), (0.20, '--')]:
    c_bl, cp_bl = budget_line(y=y, yp=yp, t=t, tp=tp, r=r_val)
    ax.plot(c_bl, cp_bl, lw=2, ls=ls, label=f'BL at $r={r_val:.0%}$')

E = endowment(y, yp, t, tp)
ax.scatter(*E, s=80, color='crimson', zorder=5, label=f'$E = {E}$ (pivot point)')
ax.set_xlim(0, 220); ax.set_ylim(0, 260)
ax.set_xlabel('Present consumption $c$'); ax.set_ylabel('Future consumption $c\'$')
ax.set_title('Interest-rate hike: budget line pivots clockwise through $E$')
ax.legend(fontsize=9)
plt.show()


## Part D — Experiment 2: shock to current income (parallel outward shift)

When present income rises from $y_1$ to $y_2$ (holding $y', t, t', r$ fixed), lifetime wealth rises by exactly $\Delta y$. The budget line shifts **parallel outward** (slope unchanged) and the endowment $E$ slides **horizontally right**.

With $r$ fixed there is **no substitution effect** — only a pure positive income effect. Both $c$ and $c'$ rise. Because $c$ rises by *less* than $\Delta y$ (the consumer saves part of the windfall to boost future consumption), **savings rise too**. This is **consumption smoothing** in action.


In [ ]:
fig, ax = plt.subplots()

# Shared parameters; only y changes across the two scenarios.
yp, t, tp, r, beta = 120, 10, 15, 0.10, 1 / 1.10

scenarios = [(100, '_1', '-'), (130, '_2', '--')]
for y_val, suf, ls in scenarios:
    c_bl, cp_bl = budget_line(y=y_val, yp=yp, t=t, tp=tp, r=r)
    ax.plot(c_bl, cp_bl, lw=2, ls=ls, label=f'BL at $y={y_val}$')
    E = endowment(y_val, yp, t, tp)
    c_star, cp_star = solve_optimum(y=y_val, yp=yp, t=t, tp=tp, r=r, beta=beta)
    ax.scatter(*E, s=60, zorder=5, label=f'$E{suf} = {E}$')
    ax.scatter(c_star, cp_star, s=60, marker='X', zorder=5,
               label=f'$(c^*, c\'^*){suf} = ({c_star:.1f}, {cp_star:.1f})$')

ax.set_xlim(0, 260); ax.set_ylim(0, 290)
ax.set_xlabel('Present consumption $c$'); ax.set_ylabel('Future consumption $c\'$')
ax.set_title('Rise in current income: parallel outward BL shift')
ax.legend(fontsize=8)
plt.show()

# Numerical check: how much of the income increase shows up in each period?
c1, cp1 = solve_optimum(y=100, yp=yp, t=t, tp=tp, r=r, beta=beta)
c2, cp2 = solve_optimum(y=130, yp=yp, t=t, tp=tp, r=r, beta=beta)
dy  = 130 - 100
print(f'Income change delta y   = {dy}')
print(f'Change in c  (c2 - c1)  = {c2 - c1:.2f}  ({100 * (c2 - c1) / dy:.0f}% of dy)')
print(f"Change in c' (cp2-cp1)  = {cp2 - cp1:.2f}")
print(f'Saving rose by          ~ {(dy - (c2 - c1)):.2f}  (= change in s)')


## Part E — Experiment 3: PV-neutral tax-timing swap (Ricardian Equivalence)

Suppose the government **raises present taxes by $\Delta t$** and **cuts future taxes by $(1+r)\Delta t$**, keeping the present value of the tax bill constant. Then lifetime wealth $we$ is unchanged, the budget line is unchanged, and the optimum $(c^*, c'^*)$ is unchanged.

What *does* move is the endowment point — it slides *along* the fixed budget line toward the upper-left. Savings fall by exactly $\Delta t$ to finance the forced transfer: the consumer pays the extra tax today out of savings and gets the refund tomorrow.

This is **Ricardian Equivalence**: the timing of lump-sum taxes doesn't matter for consumption or welfare when $we$ is unchanged.


In [ ]:
params = dict(y=100, yp=120, r=0.10, beta=1 / 1.10)
dt = 8                                   # size of current tax hike
scen_a = dict(t=10,    tp=15,            **params)
scen_b = dict(t=10+dt, tp=15 - (1+params['r'])*dt, **params)

print('Baseline we =', lifetime_wealth(**{k: scen_a[k] for k in ['y','yp','t','tp','r']}))
print('New      we =', lifetime_wealth(**{k: scen_b[k] for k in ['y','yp','t','tp','r']}))
print('Baseline (c*, c\'*) =', solve_optimum(**scen_a))
print('New      (c*, c\'*) =', solve_optimum(**scen_b))

fig, ax = plt.subplots()
c_bl, cp_bl = budget_line(**{k: scen_a[k] for k in ['y','yp','t','tp','r']})
ax.plot(c_bl, cp_bl, lw=2, label='Budget line (unchanged by the swap)')
E_a = endowment(scen_a['y'], scen_a['yp'], scen_a['t'], scen_a['tp'])
E_b = endowment(scen_b['y'], scen_b['yp'], scen_b['t'], scen_b['tp'])
ax.scatter(*E_a, s=70, color='crimson', zorder=5, label=f'$E_{{before}} = {E_a}$')
ax.scatter(*E_b, s=70, color='darkorange', zorder=5,
           label=f'$E_{{after}} = ({E_b[0]:.1f}, {E_b[1]:.1f})$')
ax.annotate('', xy=E_b, xytext=E_a,
            arrowprops=dict(arrowstyle='->', color='gray'))
c_star, cp_star = solve_optimum(**scen_a)
ax.scatter(c_star, cp_star, s=80, marker='X', color='forestgreen', zorder=5,
           label='Optimum (unchanged)')
ax.set_xlim(0, c_bl.max() * 1.05); ax.set_ylim(0, cp_bl.max() * 1.05)
ax.set_xlabel('Present consumption $c$'); ax.set_ylabel('Future consumption $c\'$')
ax.set_title('Ricardian Equivalence: $E$ slides, $(c^*, c\'^*)$ does not')
ax.legend(fontsize=8)
plt.show()


## Part F — Experiment 4: credit market imperfections (kinked budget line)

In reality banks charge a spread: the **borrowing rate** $r^b$ is higher than the **lending rate** $r^l$. The budget line becomes **kinked at the endowment point $E$** — flatter to the left of $E$ (saver branch, slope $-(1+r^l)$) and steeper to the right (borrower branch, slope $-(1+r^b)$).

Under this friction, a PV-neutral tax-timing swap is no longer neutral for a borrower: the government effectively lends them funds at the cheaper rate $r^l$ rather than the market borrowing rate $r^b$. **Ricardian Equivalence breaks down** — a current tax cut financed by higher future taxes genuinely makes a credit-constrained borrower better off.


In [ ]:
def kinked_budget(E, rl, rb, span=80):
    """Draw a kinked BL: slope -(1+rl) to the left of E, -(1+rb) to the right."""
    c_lend   = np.linspace(max(0, E[0] - span), E[0], 100)
    cp_lend  = E[1] + (1 + rl) * (E[0] - c_lend)
    c_borrow = np.linspace(E[0], E[0] + span, 100)
    cp_borrow = E[1] - (1 + rb) * (c_borrow - E[0])
    return c_lend, cp_lend, c_borrow, cp_borrow

y, yp = 100, 120
t_orig, t_cut = 10, 4
rl, rb = 0.05, 0.20                       # big spread for visibility
dt = t_orig - t_cut                       # size of the tax cut
tp_orig = 15
tp_new  = tp_orig + (1 + rb) * dt         # PV-balanced hike at the borrower's rate

E_orig = (y - t_orig, yp - tp_orig)
E_new  = (y - t_cut,  yp - tp_new)

fig, ax = plt.subplots()
cL, cpL, cB, cpB = kinked_budget(E_orig, rl, rb)
ax.plot(cL, cpL, lw=2, color='#1f77b4', label=f'BL saver ($r^l={rl:.0%}$)')
ax.plot(cB, cpB, lw=2, color='#1f77b4', ls='--', label=f'BL borrower ($r^b={rb:.0%}$)')
ax.scatter(*E_orig, s=70, color='crimson', zorder=5, label=f'$E_1 = {E_orig}$')

cL2, cpL2, cB2, cpB2 = kinked_budget(E_new, rl, rb)
ax.plot(cL2, cpL2, lw=2, color='#ff7f0e')
ax.plot(cB2, cpB2, lw=2, color='#ff7f0e', ls='--',
        label='New BL after PV-neutral tax-timing swap')
ax.scatter(*E_new, s=70, color='darkorange', zorder=5,
           label=f'$E_2 = ({E_new[0]:.1f}, {E_new[1]:.1f})$')
ax.annotate('', xy=E_new, xytext=E_orig,
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.set_xlim(0, 180); ax.set_ylim(0, 220)
ax.set_xlabel('Present consumption $c$'); ax.set_ylabel('Future consumption $c\'$')
ax.set_title('Credit market imperfections: kinked BL + tax-timing swap')
ax.legend(fontsize=8)
plt.show()


## Part G — Consumption smoothing in one line

Across every experiment above, the pattern is the same: when current income is hit by a transitory shock, the consumer uses the credit market to **split the response across both periods** instead of letting consumption move one-for-one with income.

In the numeric experiment of Part D a $\$30$ income windfall produced only a $\$15$ jump in current consumption — the other $\$15$ went into savings, lifting *future* consumption instead. If we rerun the economy many times under random income shocks and compute the volatility of each series, consumption's standard deviation would be visibly smaller than income's. That's the empirical fingerprint we'll chase in **Notebook 3** — on actual macro data.
